# Grid-based Boomerang — quick sampling demo

Sample from `make_gaussian`, `make_banana`, and `make_gaussian_mixture` (from `sazz.models.math_targets`, imported read-only) using the new `GridBoomerangSampler` (`sazz/gpu_friendly/samplers/grid_boomerang.py`), which uses the Andral & Kamatani (2024) grid-based piecewise-constant upper bound instead of Brent/PLI.

This notebook only exercises the new `sazz/gpu_friendly/` tree — nothing in `sazz/samplers/` or `sazz/models/` is modified. The analytic `math_targets.py` closures are pure functions of `beta` (no detaching), so they compose directly with the grid sampler's `torch.func.grad`/`vmap`/`jvp`-based bound construction without any adapter.

In [ ]:
import os
from pathlib import Path

import math
import numpy as np
import torch
import matplotlib.pyplot as plt

if Path.cwd().name == "notebooks":
    os.chdir("..")

from sazz.models.math_targets import make_gaussian, make_banana, make_gaussian_mixture
from sazz.gpu_friendly.samplers.grid_boomerang import GridBoomerangSampler
from sazz.gpu_friendly.samplers.grid_zigzag import GridZigZagSampler
from sazz.gpu_friendly.utils.resample import resample_zigzag_path_torch, resample_boomerang_path_torch

In [ ]:
def run_grid_boomerang(target, N=50_000, refresh_rate=1.0, n_segments=20,
                        grid_t_max_init=math.pi / 4, dtype=torch.float64):
    """Build a GridBoomerangSampler for `target` and run it for N skeleton points."""
    sampler = GridBoomerangSampler(
        grad_target=target.grad_target,
        D=target.D,
        refresh_rate=refresh_rate,
        grid_t_max_init=grid_t_max_init,
        n_segments=n_segments,
        dtype=dtype,
    )
    sampler.preprocess(x_ref=target.x_ref, Sigma_inv=target.Sigma_inv * 0.1)
    result = sampler.sample(N=N, diagnostics=True)
    result["sampler"] = sampler
    return result

def run_grid_zigzag(target, N=50_000, n_segments=20,
                        grid_t_max_init=1.0, dtype=torch.float64):
    """Build a GridBoomerangSampler for `target` and run it for N skeleton points."""
    sampler = GridZigZagSampler(
        grad_target=target.grad_target,
        D=target.D,
        gamma=0.01,
        grid_t_max_init=grid_t_max_init,
        n_segments=n_segments,
        dtype=dtype,
    )
    result = sampler.sample(N=N, x0=target.x_ref, diagnostics=True)
    result["sampler"] = sampler
    return result

def plot_marginals(target, result, model="zigzag", max_coords=4, bins=60, burnin_frac=0.5):
    """Histogram of each coordinate vs the analytic marginal PDF, plus the
    adaptive grid_t_max trajectory over the run."""
    coords = list(target.marginal_grids.keys())[:max_coords]
    # Resamplers require torch.Tensor inputs (they read .dtype/.device off
    # them directly) -- keep these as tensors here, only convert to numpy
    # for matplotlib after resampling.
    positions = result["positions"]
    velocities = result["velocities"]
    times = result["times"]
    n = positions.shape[0]
    if model == "zigzag":
        samples = resample_zigzag_path_torch(positions, velocities, times, N_resample=10_000, burnin_frac=burnin_frac)
    elif model == "boomerang":
        samples = resample_boomerang_path_torch(positions, velocities, times, target.x_ref, N_resample=10_000, burnin_frac=burnin_frac)
    else:
        raise ValueError(f"Please provide a valid model ('zigzag' or 'boomerang'), got {model!r}")
    samples = samples.numpy()

    fig, axes = plt.subplots(1, len(coords) + 1, figsize=(3.2 * (len(coords) + 1), 3))

    for i, c in enumerate(coords):
        ax = axes[i]
        info = target.marginal_grids[c]
        ax.hist(samples[:, c], bins=bins, density=True, alpha=0.5, label="grid boomerang")
        ax.plot(info["grid"], info["pdf"], "k--", lw=1.5, label="analytic")
        ax.set_title(info["label"])
        if i == 0:
            ax.legend(fontsize=8)

    ax = axes[-1]
    ax.plot(result["grid_t_max_log"])
    ax.set_title("adaptive grid_t_max")
    ax.set_xlabel("iteration")

    fig.tight_layout()
    plt.show()

    print(f"bound_violations: {result['bound_violations']}")
    print(f"gradient_evals: {result['gradient_evals']} "
          f"({result['gradient_evals'] / n:.1f} / skeleton point)")

In [ ]:
def plot_marginals_joint(target, result_list, max_coords=4, bins=60, burnin_frac=0.5):
    """Histogram of each coordinate vs the analytic marginal PDF, with
    zigzag and boomerang samples overlaid on the same axes."""
    coords = list(target.marginal_grids.keys())[:max_coords]

    fig, axes = plt.subplots(1, len(coords), figsize=(3.2 * len(coords), 3))
    if len(coords) == 1:
        axes = [axes]

    for result in result_list:
        # Resamplers require torch.Tensor inputs (they read .dtype/.device
        # off them directly) -- keep these as tensors here, only convert to
        # numpy for matplotlib after resampling.
        positions = result["positions"]
        velocities = result["velocities"]
        times = result["times"]

        if isinstance(result["sampler"], GridZigZagSampler):
            samples = resample_zigzag_path_torch(positions, velocities, times, N_resample=10_000, burnin_frac=burnin_frac)
            label = "Zigzag"
        elif isinstance(result["sampler"], GridBoomerangSampler):
            samples = resample_boomerang_path_torch(positions, velocities, times, target.x_ref, N_resample=10_000, burnin_frac=burnin_frac)
            label = "Boomerang"
        else:
            raise ValueError(f"Unrecognized sampler type: {type(result['sampler'])!r}")
        samples = samples.numpy()

        for i, c in enumerate(coords):
            axes[i].hist(samples[:, c], bins=bins, density=True, alpha=0.5, label=label)

        n = positions.shape[0]
        print(f"[{label}] bound_violations: {result['bound_violations']}")
        print(f"[{label}] gradient_evals: {result['gradient_evals']} "
              f"({result['gradient_evals'] / n:.1f} / skeleton point)")

    for i, c in enumerate(coords):
        info = target.marginal_grids[c]
        axes[i].plot(info["grid"], info["pdf"], "k--", lw=1.5, label="Analytic")
        axes[i].set_title(info["label"])
        if i == 0:
            axes[i].legend(fontsize=8)

    fig.tight_layout()
    fig.savefig(fname=f"results/plots/math_targets/{target.name}.png")
    plt.show()


## BANANA

In [ ]:

target_banana_original = make_banana(a=1.0, scale=2.0)

warmup_banana_zigzag_small = run_grid_zigzag(target_banana_original, N=2_000)
warmup_banana_zigzag_medium = run_grid_zigzag(target_banana_original, N=5_000)
warmup_banana_zigzag_large = run_grid_zigzag(target_banana_original, N=10_000)

In [ ]:
warmup_banana_zigzag_very_large = run_grid_zigzag(target_banana_original, N=50_000)

## Covariance from a PDMP skeleton

The invariant measure is the path's occupation measure, so estimators are
**time averages**, not averages over skeleton points:

$$\mu = \frac{1}{T}\int_0^T x(t)\,dt,\qquad
M = \frac{1}{T}\int_0^T x(t)x(t)^\top dt,\qquad
\Sigma = M - \mu\mu^\top$$

**ZigZag.** Between events $x(t)=x_i+v_i s$, $s\in[0,\Delta_i]$, so each
segment integrates exactly:

$$\int_0^{\Delta}(x_i+v_is)\,ds=\Delta x_i+\tfrac12\Delta^2 v_i$$

$$\int_0^{\Delta}(x_i+v_is)(x_i+v_is)^\top ds
=\Delta\,x_ix_i^\top+\tfrac12\Delta^2\!\left(x_iv_i^\top+v_ix_i^\top\right)
+\tfrac13\Delta^3\,v_iv_i^\top$$

**Boomerang.** Free flight is elliptical about $x_{\mathrm{ref}}$:

$$x(s)=x_{\mathrm{ref}}+(x_i-x_{\mathrm{ref}})\cos s+v_i\sin s$$

Closed forms exist but need $\Sigma_{\mathrm{ref}}^{1/2}$, i.e. the old
$\Sigma$ to estimate the new one. Gauss–Legendre quadrature per segment
avoids that circularity and is exact to machine precision on smooth arcs.


In [ ]:
import numpy as np
import torch

def skeleton_covariance(positions, velocities, times,
                        burnin_frac=0.2):
    """Time-averaged mean and covariance from a PDMP skeleton.

    The skeleton is a continuous path, and the invariant measure is its
    occupation measure over time. Averaging over skeleton POINTS weights a
    1e-4 segment the same as a 1.0 segment, which biases toward wherever
    events cluster -- exactly the high-rate regions. These are time averages.

    Returns (mu [D], Sigma [D, D], T) with T the integrated path time.
    """
    pos, vel, t = positions, velocities, times
    n = pos.shape[0]
    i0 = int(burnin_frac * (n - 1))
    pos, vel, t = pos[i0:], vel[i0:], t[i0:]

    dt = (t[1:] - t[:-1]).unsqueeze(-1)          # [S, 1]
    x0, v0 = pos[:-1], vel[:-1]                  # [S, D]
    T = float(dt.sum())
    if T <= 0:
        raise ValueError("non-positive total path time")

    # -- ZigZag: exact segment integrals --
    d1, d2, d3 = dt, dt ** 2, dt ** 3
    int_x = d1 * x0 + 0.5 * d2 * v0                       # [S, D]
    xx = torch.einsum("si,sj->sij", x0, x0)
    xv = torch.einsum("si,sj->sij", x0, v0)
    vv = torch.einsum("si,sj->sij", v0, v0)
    d1e, d2e, d3e = d1.unsqueeze(-1), d2.unsqueeze(-1), d3.unsqueeze(-1)
    int_xx = d1e * xx + 0.5 * d2e * (xv + xv.transpose(1, 2)) + (d3e / 3.0) * vv

    mu = int_x.sum(dim=0) / T
    M = int_xx.sum(dim=0) / T
    Sigma = M - torch.outer(mu, mu)
    Sigma = 0.5 * (Sigma + Sigma.T)   # kill asymmetry from roundoff
    return mu, Sigma, T



In [ ]:
mu_small, Sigma_zz_small, T_small = skeleton_covariance(warmup_banana_zigzag_small['positions'], 
                    warmup_banana_zigzag_small['velocities'],
                    warmup_banana_zigzag_small['times'])

mu, Sigma_zz, T = skeleton_covariance(warmup_banana_zigzag_medium['positions'], 
                    warmup_banana_zigzag_medium['velocities'],
                    warmup_banana_zigzag_medium['times'])

mu_large, Sigma_zz_large, T_large = skeleton_covariance(warmup_banana_zigzag_large['positions'], 
                    warmup_banana_zigzag_large['velocities'],
                    warmup_banana_zigzag_large['times'])

Sigma_inv_zz_small = torch.linalg.inv(Sigma_zz_small)
Sigma_inv_zz = torch.linalg.inv(Sigma_zz)
Sigma_inv_zz_large = torch.linalg.inv(Sigma_zz_large)

In [ ]:
mu_very_large, Sigma_zz_very_large, T_very_large = skeleton_covariance(warmup_banana_zigzag_very_large['positions'], 
                    warmup_banana_zigzag_very_large['velocities'],
                    warmup_banana_zigzag_very_large['times'])

Sigma_inv_zz_very_large = torch.linalg.inv(Sigma_zz_very_large)

In [ ]:
mu_end, Sigma_zz_end, T_end = skeleton_covariance(warmup_banana_zigzag_very_large['positions'][40_000:], 
                    warmup_banana_zigzag_very_large['velocities'][40_000:],
                    warmup_banana_zigzag_very_large['times'][40_000:])

Sigma_inv_zz_end = torch.linalg.inv(Sigma_zz_end)

In [ ]:
print("2K skeletons:", Sigma_inv_zz_small, "\n")
print("5K skeletons:", Sigma_inv_zz, "\n")
print("10K skeletons:", Sigma_inv_zz_large, "\n")
print("50K skeletons:", Sigma_inv_zz_very_large, "\n")
print("last 10K of 50K skeletons:", Sigma_inv_zz_end, "\n")

In [ ]:
from scipy import integrate
from sazz.models.model import TorchTarget

def make_banana_warm(D=2, a=1.0, scale=1.0, sigma_inv_scale=1.0, warm_Sigma_inv=None, dtype=torch.float64):
    """
    Banana E(b0,b1) = 0.5*(b0/scale)^2 + 0.5*(b1 - a*(b0/scale)^2)^2.
    """
    if D != 2:
        raise NotImplementedError("Only D=2 supported.")

    def grad_target(beta):
        b0, b1 = beta[0], beta[1]
        u = b0 / scale
        r = b1 - a * u**2
        dE_db0 = u / scale - 2.0 * a * u / scale * r
        dE_db1 = r
        return torch.stack([dE_db0, dE_db1])

    s2 = scale**2
    Z = scale * 2 * np.pi
    grid_0 = np.linspace(-4 * scale, 4 * scale, 500)
    grid_1 = np.linspace(-4, 4 + 16 * a, 500)
    marg_0 = np.exp(-0.5 * grid_0**2 / s2) / (scale * np.sqrt(2 * np.pi))

    def unnorm_joint(b0, b1):
        u = b0 / scale
        return np.exp(-0.5 * u**2 - 0.5 * (b1 - a * u**2)**2)

    marg_1 = np.zeros_like(grid_1)
    b0_lim = 8 * scale
    for i, b1_val in enumerate(grid_1):
        val, _ = integrate.quad(lambda b0: unnorm_joint(b0, b1_val), -b0_lim, b0_lim)
        marg_1[i] = val / Z

    marginal_grids = {
        0: {"grid": grid_0, "pdf": marg_0, "label": r"$\beta_1$"},
        1: {"grid": grid_1, "pdf": marg_1, "label": r"$\beta_2$"},
    }
    if warm_Sigma_inv is None:
        precision = (sigma_inv_scale) * torch.diag(torch.tensor([1.0, 1.0 / 3.0], dtype=dtype))
    else:
        precision = warm_Sigma_inv

    return TorchTarget(
        name=f"banana_D{D}_a{a}_s{scale}",
        D=D,
        grad_target=grad_target,
        x_ref=torch.tensor([0.0, a], dtype=dtype),
        Sigma_inv=precision,
        marginal_grids=marginal_grids,
    )


In [ ]:
target_banana_warm = make_banana_warm(a=1.0, scale=2.0, warm_Sigma_inv=Sigma_inv_zz)

banana_zigzag = run_grid_zigzag(target_banana_warm, N=50_000)
banana_boomerang = run_grid_boomerang(target_banana_warm, N=50_000)

In [ ]:
a, scale = 1.0, 2.0

positions_zigzag = banana_zigzag["positions"].numpy()
positions_boom = banana_boomerang["positions"].numpy()

pad = 1.5
all_b0 = np.concatenate([positions_zigzag[:, 0], positions_boom[:, 0]])
all_b1 = np.concatenate([positions_zigzag[:, 1], positions_boom[:, 1]])
b0_grid = np.linspace(all_b0.min() - pad, all_b0.max() + pad, 300)
b1_grid = np.linspace(all_b1.min() - pad, all_b1.max() + pad, 300)
B0, B1 = np.meshgrid(b0_grid, b1_grid)
U = B0 / scale
E = 0.5 * U**2 + 0.5 * (B1 - a * U**2) ** 2
density = np.exp(-E)

# Log-spaced DENSITY levels: naturally denser near the peak (where density
# varies fast) and still reaches out into the tails, unlike either linear
# density levels (collapse near zero) or linear -E levels (peak
# under-resolved). Adjust the low end (1e-4 here) to pull the outermost
# ring further out/in.
levels = np.logspace(np.log10(density.max()) - 4, np.log10(density.max()), 12)

fig, ax = plt.subplots(figsize=(4, 4))
ax.contour(B0, B1, density, levels=levels, colors="black", linewidths=0.6, alpha=0.6)
ax.scatter(positions_zigzag[:, 0], positions_zigzag[:, 1], s=2, alpha=0.3, label="Zigzag")
ax.scatter(positions_boom[:, 0], positions_boom[:, 1], s=2, alpha=0.3, label="Boomerang")
#ax.set_title("Banana — skeleton")
ax.legend(fontsize=8)
fig.savefig(fname="results/plots/math_targets/banana_contour.png")
plt.show()


In [ ]:
target_banana_original.Sigma_inv

In [ ]:
target_banana_warm.Sigma_inv